A notebook to validate the starting steps to port [Edge0](https://arxiv.org/abs/2609.18063) to CUDA.

Clone the fork project.

In [ ]:
!git clone https://github.com/virtualramblas/Edge0-CUDA.git
%cd Edge0-CUDA

Implement a Colab-compatible GDS Fallback Loader. It uses Direct/POSIX DMA-aligned host transfers to validate the exact memory layouts and offsets when running without local hardware NVMe controllers.

In [ ]:
%%writefile src/edge0/backends/cuda/c_src/cufile_mock_loader.cpp
#include <fcntl.h>
#include <unistd.h>
#include <iostream>
#include <cstring>
#include <torch/extension.h>
#include <cuda_runtime.h>

class GDSNativeLoader {
private:
    int fd_;
    std::string filepath_;

public:
    GDSNativeLoader(const std::string& filepath) : filepath_(filepath) {
        fd_ = open(filepath.c_str(), O_RDONLY);
        if (fd_ < 0) {
            throw std::runtime_error("Could not open file: " + filepath);
        }
    }

    void register_gpu_buffer(torch::Tensor dev_tensor) {
        // Mock buffer registration for cuFileBufRegister
        TORCH_CHECK(dev_tensor.is_cuda(), "Buffer must reside on CUDA device");
    }

    int64_t load_expert(torch::Tensor target_tensor, int64_t file_offset, int64_t bytes_to_read) {
        TORCH_CHECK(target_tensor.is_cuda(), "Target tensor must be on GPU");

        // Aligned CPU buffer for DMA simulation
        void* host_buf = nullptr;
        posix_memalign(&host_buf, 4096, bytes_to_read);

        ssize_t read_bytes = pread(fd_, host_buf, bytes_to_read, file_offset);
        if (read_bytes > 0) {
            cudaMemcpy(target_tensor.data_ptr(), host_buf, read_bytes, cudaMemcpyHostToDevice);
        }
        free(host_buf);
        return read_bytes;
    }

    ~GDSNativeLoader() {
        if (fd_ >= 0) close(fd_);
    }
};

// PyBind11 bindings
PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    py::class_<GDSNativeLoader>(m, "GDSNativeLoader")
        .def(py::init<const std::string&>())
        .def("register_gpu_buffer", &GDSNativeLoader::register_gpu_buffer)
        .def("load_expert", &GDSNativeLoader::load_expert);
}

Check GPU type and CUDA Toolkit availability.

In [ ]:
!nvidia-smi
!nvcc --version

Install dependencies: PyTorch, CUTLASS headers, ninja for fast compilation.

In [ ]:
!pip install --upgrade pip ninja pytest tabulate
!apt-get update && apt-get install -y libaio-dev

In [ ]:
!pip install kvikio-cu13

Clone CUTLASS headers (v3.5.0 recommended for SM80/SM90 EVT epilogues).

In [ ]:
!git clone --depth 1 --branch v3.5.0 https://github.com/NVIDIA/cutlass.git /content/cutlass
%env CUTLASS_PATH=/content/cutlass/include

Build and install the extension inside Colab.

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=61.0.0", "wheel", "torch"]
build-backend = "setuptools.build_meta"

In [ ]:
%%writefile setup.py
import os
import torch
from setuptools import setup, find_packages
from torch.utils.cpp_extension import BuildExtension, CUDAExtension

# Safely resolve standard CUDA toolkit directories in Colab
cuda_home = os.environ.get("CUDA_HOME", "/usr/local/cuda")
cuda_include = os.path.join(cuda_home, "include")

ext_modules = [
    CUDAExtension(
        name="edge0_gds_native",
        sources=["src/edge0/backends/cuda/c_src/cufile_mock_loader.cpp"],
        include_dirs=[cuda_include],  # Directly map CUDA include paths
        extra_compile_args={
            "cxx": ["-O3", "-std=c++17"],
            "nvcc": ["-O3", "-std=c++17"]
        }
    )
]

setup(
    name="edge0",
    version="0.1.0",
    packages=find_packages(where="src"),
    package_dir={"": "src"},
    ext_modules=ext_modules,
    cmdclass={"build_ext": BuildExtension}
)

Patch backend to skip the MLX requirements (only CUDA here :)).

In [ ]:
import os

# 1. Patch edge0/backends/__init__.py to make mlx imports optional
backends_init_path = "src/edge0/backends/__init__.py"

# Read the current file content if it exists
if os.path.exists(backends_init_path):
    with open(backends_init_path, "r") as f:
        content = f.read()
else:
    content = ""

# Rewrite backends/__init__.py with dynamic platform safety checks
fused_backends_init = """import platform

# Always export base structures
from edge0.backends.base import BaseBackend, BaseExpertCache

# Conditionally import backends based on platform compatibility
try:
    if platform.system() == "Darwin":
        from edge0.backends.mlx.backend import MLXBackend
        from edge0.backends.mlx.moe_layer import MLXMoELayer
except ImportError:
    pass

try:
    # Always attempt importing CUDA on Linux/Windows
    from edge0.backends.cuda.backend import CudaBackend
    from edge0.backends.cuda.cache import GDSStreamingExpertCache
    from edge0.backends.cuda.moe_layer import CUTLASSGDSMoE
except ImportError:
    pass
"""

with open(backends_init_path, "w") as f:
    f.write(fused_backends_init)
print(f"✅ Patched: {backends_init_path} with OS-safe conditional imports.")

In [ ]:
%%writefile src/edge0/backends/base.py
import abc
from typing import Optional, List, Tuple
import torch
import torch.nn as nn

class BaseBackend(abc.ABC):
    """Abstract device execution backend managing streams and memory."""
    @abc.abstractmethod
    def synchronize(self):
        """Wait for all pending device compute and transfer operations to finish."""
        pass


class BaseExpertCache(abc.ABC):
    """Abstract expert cache managing VRAM slots and asynchronous transfers."""
    @abc.abstractmethod
    def wait_for_expert(self, expert_id: int) -> torch.Tensor:
        """Resolve an in-flight transfer and return resident expert weights."""
        pass


class BaseMoELayer(nn.Module, abc.ABC):
    """
    Abstract Mixture-of-Experts layer interface.
    Enforces a consistent forward pass and lookahead routing hook
    across MLX and CUDA backends.
    """
    def __init__(
        self,
        hidden_dim: int,
        intermediate_dim: int,
        num_experts: int,
        top_k: int,
        backend: Optional[BaseBackend] = None
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.intermediate_dim = intermediate_dim
        self.num_experts = num_experts
        self.top_k = top_k
        self.backend = backend

    @abc.abstractmethod
    def forward(
        self,
        x: torch.Tensor,
        predicted_routing_tokens: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """Forward computation for the MoE layer."""
        pass

    def predict_experts(self, hidden_states: torch.Tensor) -> List[int]:
        """
        Optional prerouter lookahead hook to predict active experts
        1-2 layers ahead of the current execution.
        """
        # Default fallback: return empty list if not overridden
        return []

Force a clean, standard (non-editable) install with verbose logs.

In [ ]:
!pip install . --no-build-isolation -v

Create the automated test suite.

In [ ]:
%cd src/edge0
!mkdir tests

In [ ]:
%%writefile tests/test_cuda_verification.py
import pytest
import os
import torch
import numpy as np
from edge0.backends.cuda.cache import GDSStreamingExpertCache
from edge0.backends.cuda.moe_layer import CUTLASSGDSMoE

TMP_WEIGHT_FILE = "/tmp/mock_nvme_weights.bin"

@pytest.fixture(scope="session", autouse=True)
def prepare_mock_nvme_file():
    """Generates a dummy binary file to act as the NVMe storage device."""
    num_experts = 4
    hidden_dim = 128
    intermediate_dim = 256

    # Generate random weights in float16
    total_elements = num_experts * hidden_dim * intermediate_dim
    weights = np.random.randn(total_elements).astype(np.float16)

    with open(TMP_WEIGHT_FILE, "wb") as f:
        f.write(weights.tobytes())

    yield

    if os.path.exists(TMP_WEIGHT_FILE):
        os.remove(TMP_WEIGHT_FILE)

def test_c_extension_compilation():
    """Verifies that the compiled C++/CUDA GDS loader loads and instantiates without error."""
    import edge0_gds_native
    loader = edge0_gds_native.GDSNativeLoader(TMP_WEIGHT_FILE)
    assert loader is not None, "Failed to instantiate GDSNativeLoader"

def test_gds_cache_direct_loading():
    """Tests that GDSStreamingExpertCache correctly streams expert weights directly to GPU."""
    hidden_dim = 128
    intermediate_dim = 256
    expert_shape = (hidden_dim, intermediate_dim)

    cache = GDSStreamingExpertCache(
        num_slots=2,
        expert_shape=expert_shape,
        file_path=TMP_WEIGHT_FILE,
        device="cuda:0"
    )

    # 1 expert = 128 * 256 * 2 bytes = 65536 bytes
    bytes_per_exp = hidden_dim * intermediate_dim * 2
    cache.register_expert_offset(expert_id=0, offset=0, num_bytes=bytes_per_exp)
    cache.register_expert_offset(expert_id=1, offset=bytes_per_exp, num_bytes=bytes_per_exp)

    # Trigger load and verify
    cache.trigger_prerouter_load(expert_id=0)
    w0 = cache.wait_for_expert(expert_id=0)

    assert w0.is_cuda, "Loaded tensor is not resident in GPU memory"
    assert w0.shape == expert_shape, f"Shape mismatch: {w0.shape} vs {expert_shape}"
    assert not torch.all(w0 == 0), "Weights were not read from storage (remained all zeros)"

def test_moe_forward_and_lora_execution():
    """Executes a full forward pass through the CUTLASS + GDS MoE layer."""
    hidden_dim = 128
    intermediate_dim = 256
    num_experts = 4
    top_k = 2
    seq_len = 8

    cache = GDSStreamingExpertCache(
        num_slots=4,
        expert_shape=(hidden_dim, intermediate_dim),
        file_path=TMP_WEIGHT_FILE,
        device="cuda:0"
    )
    bytes_per_exp = hidden_dim * intermediate_dim * 2
    for e in range(num_experts):
        cache.register_expert_offset(e, e * bytes_per_exp, bytes_per_exp)
        cache.trigger_prerouter_load(e)

    layer = CUTLASSGDSMoE(
        hidden_dim=hidden_dim,
        intermediate_dim=intermediate_dim,
        num_experts=num_experts,
        top_k=top_k,
        gds_cache=cache,
        lora_rank=16,
        device="cuda:0"
    )

    dummy_input = torch.randn(1, seq_len, hidden_dim, device="cuda:0", dtype=torch.float32)
    output = layer(dummy_input)

    assert output.shape == (1, seq_len, intermediate_dim), f"Unexpected output shape: {output.shape}"
    assert not torch.isnan(output).any(), "MoE forward output contains NaNs"
    assert not torch.isinf(output).any(), "MoE forward output contains Infs"

Execute the test suite.

In [ ]:
%cd /content/Edge0-CUDA/
!pytest src/edge0/tests/test_cuda_verification.py -v -s